# Blue Book Phenomenology: Analysis

Second-phase notebook. The OCR pipeline + LLM cleaning has produced structured JSON for each case (metadata + narrative). This notebook:

1. Embeds LLM-cleaned narratives with nomic-embed-text-v1.5
2. UMAP + HDBSCAN clustering
3. Visualization (original vs narrative embeddings)
4. PCA analysis
5. BERTopic topic extraction
6. Fine structure exploration (K-means, higher PCA, HDBSCAN variants)
7. Bulk load to Supabase via direct Postgres
8. Structured feature extraction (Batch API)
9. Feature extraction retrieval

**Requires:** GPU runtime (A100 preferred) for Cells 1-2. Other cells run on any runtime.

**API keys:** Stored in Colab Secrets tab (key icon, left sidebar):
- `ANTHROPIC_API_KEY`
- `SUPABASE_URL`
- `SUPABASE_KEY`
- `SUPABASE_DB_PASSWORD`

**Supabase project:** Blue Book Phenomenology (`kikhtagzasmscshaexra`)

## Cell 1: Setup

In [ ]:
!pip install supabase sentence-transformers einops umap-learn hdbscan polars pyarrow scikit-learn psycopg2-binary anthropic -q

from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import json
import re
import numpy as np
from tqdm import tqdm

PROJECT_DIR = "/content/drive/MyDrive/blue_book_phenomenology"
LLM_CLEAN_DIR = f"{PROJECT_DIR}/corpus_llm_clean"
FEATURES_DIR = f"{PROJECT_DIR}/corpus_features"
FINAL_CORPUS_DIR = f"{PROJECT_DIR}/corpus"
DATA_DIR = f"{PROJECT_DIR}/results/data"
MODELS_DIR = f"{PROJECT_DIR}/results/models"
VIZ_DIR = f"{PROJECT_DIR}/results/visualizations"

for d in [FEATURES_DIR, MODELS_DIR, VIZ_DIR]:
    os.makedirs(d, exist_ok=True)

with open(f"{DATA_DIR}/full_cluster_results.json") as f:
    results = json.load(f)
print(f"Loaded {len(results)} case records")
print(f"LLM-cleaned JSON: {len([f for f in os.listdir(LLM_CLEAN_DIR) if f.endswith('.json')])}")

## Cell 2: Embed LLM-Cleaned Narratives

Embeds the narrative field from each LLM-cleaned JSON file with nomic-embed-text-v1.5 (8192 token context). Requires GPU. Skip if `narrative_embeddings_nomic.npy` already exists on Drive.

In [ ]:
if os.path.exists(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy"):
    print("Narrative embeddings already exist. Loading...")
    embeddings = np.load(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy")
    print(f"Loaded: {embeddings.shape}")
else:
    import torch
    from sentence_transformers import SentenceTransformer

    print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

    print("Loading narratives...")
    narratives = []
    filenames = []
    metadata_list = []
    for r in tqdm(results, desc="Reading"):
        fname = r['filename']
        json_path = os.path.join(LLM_CLEAN_DIR, fname + '.json')
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)
            narratives.append(data.get('narrative', ''))
            metadata_list.append(data.get('metadata', {}))
        else:
            raw_path = os.path.join(LLM_CLEAN_DIR, fname + '.raw.md')
            if os.path.exists(raw_path):
                with open(raw_path, 'r') as f:
                    narratives.append(f.read())
                metadata_list.append({})
            else:
                narratives.append('')
                metadata_list.append({})
        filenames.append(fname)

    print(f"Loaded {len(narratives)} narratives ({sum(1 for n in narratives if len(n) > 50)} non-empty)")

    with open(f"{DATA_DIR}/llm_metadata.json", 'w') as f:
        json.dump([{"filename": fn, "metadata": md} for fn, md in zip(filenames, metadata_list)], f, indent=2)

    print("\nLoading nomic-embed-text-v1.5...")
    model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)
    prefixed = [f'search_document: {n}' for n in narratives]
    embeddings = model.encode(prefixed, batch_size=16, show_progress_bar=True,
                              device='cuda' if torch.cuda.is_available() else 'cpu')
    np.save(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy", embeddings)
    print(f"Saved: {embeddings.shape}")

    import umap, hdbscan
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    coords = reducer.fit_transform(embeddings)
    np.save(f"{FINAL_CORPUS_DIR}/narrative_umap_coords.npy", coords)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean', cluster_selection_method='eom')
    labels = clusterer.fit_predict(coords)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = (labels == -1).sum() / len(labels) * 100
    print(f"Found {n_clusters} clusters ({noise_pct:.1f}% noise)")
    np.save(f"{FINAL_CORPUS_DIR}/narrative_cluster_labels.npy", labels)

    narrative_results = [{"filename": filenames[i], "x": float(coords[i, 0]), "y": float(coords[i, 1]),
                          "cluster": int(labels[i]), "narrative_length": len(narratives[i])}
                         for i in range(len(filenames))]
    with open(f"{DATA_DIR}/narrative_cluster_results.json", 'w') as f:
        json.dump(narrative_results, f, indent=2)
    print("All results saved.")

## Cell 3: Visualization — Original vs Narrative

Side-by-side UMAP comparison.

In [ ]:
import matplotlib.pyplot as plt

orig_coords = np.load(f"{FINAL_CORPUS_DIR}/umap_coords.npy")
orig_labels = np.load(f"{FINAL_CORPUS_DIR}/cluster_labels.npy")
narr_coords = np.load(f"{FINAL_CORPUS_DIR}/narrative_umap_coords.npy")
narr_labels = np.load(f"{FINAL_CORPUS_DIR}/narrative_cluster_labels.npy")

n_orig = len(set(orig_labels)) - (1 if -1 in orig_labels else 0)
noise_orig = (orig_labels == -1).sum() / len(orig_labels) * 100
n_narr = len(set(narr_labels)) - (1 if -1 in narr_labels else 0)
noise_narr = (narr_labels == -1).sum() / len(narr_labels) * 100

fig, axes = plt.subplots(1, 2, figsize=(28, 12))
cmap = plt.cm.tab20(np.linspace(0, 1, 20))

ax = axes[0]
noise_mask = orig_labels == -1
ax.scatter(orig_coords[noise_mask, 0], orig_coords[noise_mask, 1], c='lightgray', alpha=0.3, s=8)
for i, cl in enumerate(sorted(set(orig_labels) - {-1})):
    mask = orig_labels == cl
    ax.scatter(orig_coords[mask, 0], orig_coords[mask, 1], c=[cmap[i % 20]], alpha=0.7, s=12)
ax.set_title(f"Original (full document) \u2014 {n_orig} clusters, {noise_orig:.0f}% noise\n(geographic/administrative)", fontsize=14)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

ax = axes[1]
noise_mask = narr_labels == -1
ax.scatter(narr_coords[noise_mask, 0], narr_coords[noise_mask, 1], c='lightgray', alpha=0.3, s=8)
for i, cl in enumerate(sorted(set(narr_labels) - {-1})):
    mask = narr_labels == cl
    ax.scatter(narr_coords[mask, 0], narr_coords[mask, 1], c=[cmap[i % 20]], alpha=0.7, s=12)
ax.set_title(f"Narrative only (LLM-cleaned) \u2014 {n_narr} clusters, {noise_narr:.0f}% noise", fontsize=14)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/original_vs_narrative.png", dpi=200)
plt.show()

## Cell 4: PCA Analysis

What dimensions of variation structure the narrative embedding space?

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

narr_embeddings = np.load(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy")
narr_labels = np.load(f"{FINAL_CORPUS_DIR}/narrative_cluster_labels.npy")

pca = PCA(n_components=20)
pca_result = pca.fit_transform(narr_embeddings)

print("Variance explained:")
cumulative = 0
for i, var in enumerate(pca.explained_variance_ratio_):
    cumulative += var
    print(f"  PC{i+1}: {var*100:.1f}% (cum: {cumulative*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 21), pca.explained_variance_ratio_ * 100)
ax.set_xlabel("Principal Component"); ax.set_ylabel("Variance Explained (%)")
ax.set_title("PCA on Narrative Embeddings")
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/pca_narrative_scree.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(14, 10))
noise_mask = narr_labels == -1
ax.scatter(pca_result[noise_mask, 0], pca_result[noise_mask, 1], c='lightgray', alpha=0.2, s=5)
cmap = plt.cm.tab20(np.linspace(0, 1, 20))
for i, cl in enumerate(sorted(set(narr_labels) - {-1})):
    mask = narr_labels == cl
    ax.scatter(pca_result[mask, 0], pca_result[mask, 1], c=[cmap[i % 20]], alpha=0.6, s=10)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("PC1 vs PC2 (colored by HDBSCAN cluster)")
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/pca_narrative_pc1_pc2.png", dpi=200)
plt.show()

print("\n--- Extremes of PC1 ---")
for label, indices in [("Low", np.argsort(pca_result[:, 0])[:5]), ("High", np.argsort(pca_result[:, 0])[-5:])]:
    print(f"{label}:")
    for idx in indices:
        print(f"  {results[idx]['filename']} ({results[idx]['location']}, {results[idx]['year']})")

## Cell 5: BERTopic on Narratives

**Note:** If transformers version conflicts, restart runtime, run Cell 1, skip to this cell.

In [ ]:
!pip install bertopic -q
from bertopic import BERTopic

narr_embeddings = np.load(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy")
narr_texts = []
for r in tqdm(results, desc="Reading"):
    json_path = os.path.join(LLM_CLEAN_DIR, r['filename'] + '.json')
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            narr_texts.append(json.load(f).get('narrative', '')[:2000])
    else:
        narr_texts.append('')

from umap import UMAP
from hdbscan import HDBSCAN
topic_model = BERTopic(
    umap_model=UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean', cluster_selection_method='eom'),
    verbose=True)
topics, probs = topic_model.fit_transform(narr_texts, narr_embeddings)

info = topic_model.get_topic_info()
print(f"\nTotal topics: {len(info)}")
for _, row in info.head(30).iterrows():
    if row['Topic'] == -1:
        print(f"\nTopic -1 (noise): {row['Count']}")
        continue
    keywords = ', '.join([w for w, _ in topic_model.get_topic(row['Topic'])[:10]])
    print(f"\nTopic {row['Topic']} ({row['Count']}): {keywords}")

topic_model.save(f"{MODELS_DIR}/bertopic_narrative")

## Cell 6: Fine Structure Exploration

Multiple clustering approaches to find structure HDBSCAN missed: HDBSCAN on full 768-dim, UMAP 10D, K-means with varying k, higher PCA components.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from collections import Counter
import hdbscan, umap
import matplotlib.pyplot as plt

embeddings = np.load(f"{FINAL_CORPUS_DIR}/narrative_embeddings_nomic.npy")
print(f"Loaded {embeddings.shape}")

# 6a: HDBSCAN on full 768-dim
print("\n=== HDBSCAN on full 768-dim ===")
for mcs in [5, 10, 20, 50]:
    labels = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3).fit_predict(embeddings)
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    noise = (labels == -1).sum()
    biggest = max(np.bincount(labels[labels >= 0])) if n_cl > 0 else 0
    print(f"  mcs={mcs}: {n_cl} clusters, {noise} noise, largest={biggest}")

# 6b: UMAP 10D + HDBSCAN
print("\n=== UMAP 10D + HDBSCAN ===")
coords_10d = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42).fit_transform(embeddings)
for mcs in [5, 10, 20, 50]:
    labels = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3).fit_predict(coords_10d)
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    noise = (labels == -1).sum()
    biggest = max(np.bincount(labels[labels >= 0])) if n_cl > 0 else 0
    print(f"  mcs={mcs}: {n_cl} clusters, {noise} noise, largest={biggest}")

# 6c: K-means
print("\n=== K-means ===")
ks = [10, 20, 30, 50, 75, 100]
sils = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    sil = silhouette_score(embeddings, labels, sample_size=5000, random_state=42)
    sizes = np.bincount(labels)
    print(f"  k={k}: sil={sil:.3f}, largest={sizes.max()}, smallest={sizes.min()}, median={int(np.median(sizes))}")
    sils.append(sil)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, sils, 'bo-')
ax.set_xlabel("k"); ax.set_ylabel("Silhouette")
ax.set_title("K-means Silhouette")
plt.savefig(f"{VIZ_DIR}/kmeans_silhouette.png", dpi=150)
plt.show()

# 6d: Best K-means
best_k = ks[np.argmax(sils)]
km_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(embeddings)
np.save(f"{FINAL_CORPUS_DIR}/narrative_kmeans_labels.npy", km_labels)

coords_2d = np.load(f"{FINAL_CORPUS_DIR}/narrative_umap_coords.npy")
fig, ax = plt.subplots(figsize=(16, 12))
cmap = plt.cm.tab20(np.linspace(0, 1, 20))
for i in range(best_k):
    mask = km_labels == i
    ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1], c=[cmap[i % 20]], alpha=0.5, s=10)
ax.set_title(f"K-means (k={best_k}) on Narrative Embeddings")
plt.savefig(f"{VIZ_DIR}/kmeans_narrative.png", dpi=200)
plt.show()

# 6e: Higher PCA components
pca = PCA(n_components=10).fit_transform(embeddings)
fig, axes = plt.subplots(2, 3, figsize=(24, 14))
for ax, (a, b) in zip(axes.flat, [(0,1),(0,2),(1,2),(2,3),(3,4),(4,5)]):
    ax.scatter(pca[:, a], pca[:, b], c=km_labels, cmap='tab20', alpha=0.3, s=5)
    ax.set_xlabel(f"PC{a+1}"); ax.set_ylabel(f"PC{b+1}")
plt.suptitle(f"PCA pairs (K-means k={best_k})")
plt.savefig(f"{VIZ_DIR}/pca_higher_components.png", dpi=200)
plt.show()

# 6f: Characterize K-means clusters
print(f"\n=== K-means k={best_k} clusters ===")
orig_map = {r['filename']: r for r in results}
for cl in range(best_k):
    members_idx = np.where(km_labels == cl)[0]
    if len(members_idx) < 5: continue
    locs = [orig_map.get(results[i]['filename'], {}).get('location', '?').split()[-1] for i in members_idx if i < len(results)]
    diversity = len(set(locs)) / max(len(locs), 1)
    top = Counter(locs).most_common(3)
    print(f"\nCluster {cl} ({len(members_idx)} cases, diversity={diversity:.2f}): {top}")
    for idx in members_idx[:3]:
        if idx < len(results): print(f"  {results[idx]['filename']}")

with open(f"{DATA_DIR}/narrative_kmeans_results.json", 'w') as f:
    json.dump([{"filename": results[i]['filename'] if i < len(results) else "", "kmeans_cluster": int(km_labels[i])}
               for i in range(len(km_labels))], f, indent=2)

## Cell 7: Bulk Load to Supabase (Direct Postgres)

Bypasses REST API. Loads LLM narratives and metadata via direct Postgres connection. Set `SUPABASE_DB_PASSWORD` in Colab Secrets.

In [ ]:
import psycopg2

db_password = userdata.get('SUPABASE_DB_PASSWORD')

conn = psycopg2.connect(
    host="aws-0-us-east-1.pooler.supabase.com",
    port=6543,
    dbname="postgres",
    user="postgres.kikhtagzasmscshaexra",
    password=db_password,
    sslmode="require"
)
conn.autocommit = False
cur = conn.cursor()
print("Connected to Postgres")

cur.execute("SELECT id, filename FROM cases")
id_map = {row[1]: row[0] for row in cur.fetchall()}
print(f"ID map: {len(id_map)} cases")

# Update extracted_narrative
print("\n--- Loading narratives ---")
updated = 0
for i, r in enumerate(tqdm(results, desc="Narratives")):
    case_id = id_map.get(r['filename'])
    if not case_id: continue
    json_path = os.path.join(LLM_CLEAN_DIR, r['filename'] + '.json')
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            narrative = json.load(f).get('narrative', '')
        if narrative:
            cur.execute("UPDATE case_text SET extracted_narrative = %s WHERE case_id = %s", (narrative, case_id))
            updated += 1
    if (i + 1) % 500 == 0:
        conn.commit()
conn.commit()
print(f"Narratives: {updated}")

# Verify
cur.execute("SELECT COUNT(*) FROM case_text WHERE extracted_narrative IS NOT NULL")
print(f"Verification: {cur.fetchone()[0]} cases have narrative")

cur.close()
conn.close()
print("Done.")

## Cell 8: Structured Feature Extraction (Batch API)

Sends each narrative to Claude Haiku to extract structured phenomenological features. Submits via Anthropic Batch API (~$5, results within 24 hours). Set `ANTHROPIC_API_KEY` in Colab Secrets.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

PROMPT = """Read this witness narrative from a Project Blue Book case file (1947-1969). Extract structured phenomenological features as a JSON object.

Return ONLY this JSON structure. Use null for features not mentioned or unclear.

{
  "shape": "disc/saucer | sphere/ball | cigar/cylinder | triangle/V | oval/egg | star-like/point | irregular | light_only | other",
  "color": "white | silver/metallic | red | orange | green | blue | yellow | multicolor | dark/black | glowing | other",
  "luminosity": "self-luminous | reflective | dark_silhouette | intermittent/flashing | color_changing | none",
  "sound": "silent | buzzing/humming | roaring | hissing | explosion | engine-like | other",
  "size_description": "witness's own size estimate or comparison",
  "estimated_altitude": "witness's altitude estimate if given",
  "movement_pattern": "stationary/hovering | straight_line | erratic/zigzag | circling | pacing_vehicle | formation | terrain_following | ascending | descending | instantaneous_direction_change | other",
  "speed_description": "witness's speed estimate or comparison",
  "interaction_with_observer": "none | approaching | retreating | pacing | circling_observer | responding_to_observer | other",
  "physical_effects": "none | electromagnetic | physiological | ground_traces | heat | odor | animal_reaction | other",
  "departure_mode": "gradual_fade | sudden_disappearance | high_speed_departure | behind_terrain | unknown",
  "witness_type": "civilian | military_ground | military_pilot | commercial_pilot | radar_operator | police | scientist | multiple_independent | other",
  "num_witnesses": "1 | 2-5 | 6-20 | 20+ | unknown",
  "duration": "seconds | 1-5_minutes | 5-30_minutes | 30+_minutes | unknown",
  "radar_confirmation": true/false/null,
  "photos_or_film": true/false/null,
  "official_conclusion": "conclusion stated in document if present",
  "brief_description": "one sentence summary of what the witness reported"
}

Return ONLY the JSON. No commentary."""

def sanitize_id(filename):
    return re.sub(r'[^a-zA-Z0-9_-]', '_', filename.replace('.json', ''))[:64]

json_files = sorted([f for f in os.listdir(LLM_CLEAN_DIR) if f.endswith('.json')])
already_done = set(f for f in os.listdir(FEATURES_DIR) if f.endswith('.json'))
to_process = [f for f in json_files if f not in already_done]
print(f"Total: {len(json_files)}, Done: {len(already_done)}, Remaining: {len(to_process)}")

requests = []
id_to_file = {}
for jf in to_process:
    with open(os.path.join(LLM_CLEAN_DIR, jf), 'r') as f:
        narrative = json.load(f).get('narrative', '')
    if len(narrative) < 50: continue
    cid = sanitize_id(jf)
    id_to_file[cid] = jf
    requests.append({"custom_id": cid, "params": {
        "model": "claude-haiku-4-5-20251001", "max_tokens": 1024,
        "messages": [{"role": "user", "content": f"{PROMPT}\n\n---\n\n{narrative[:4000]}"}]}})

print(f"Built {len(requests)} requests")
with open(f"{PROJECT_DIR}/features_batch_id_mapping.json", 'w') as f:
    json.dump(id_to_file, f)

batch = client.messages.batches.create(requests=requests)
print(f"\nBatch submitted! ID: {batch.id}, Status: {batch.processing_status}")

with open(f"{PROJECT_DIR}/features_batch_info.json", 'w') as f:
    json.dump({"batch_id": batch.id, "num_requests": len(requests), "type": "feature_extraction"}, f, indent=2)
print("Results ready within 24 hours. Run Cell 9 to retrieve.")

## Cell 9: Retrieve Feature Extraction Results

Check batch status and download structured features when ready.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

with open(f"{PROJECT_DIR}/features_batch_info.json") as f:
    batch_info = json.load(f)
with open(f"{PROJECT_DIR}/features_batch_id_mapping.json") as f:
    id_to_file = json.load(f)

batch = client.messages.batches.retrieve(batch_info['batch_id'])
print(f"Batch: {batch_info['batch_id']}")
print(f"Status: {batch.processing_status}")
print(f"Counts: {batch.request_counts}")

if batch.processing_status != 'ended':
    print("\nStill processing. Check back later.")
else:
    saved = errors = 0
    for result in client.messages.batches.results(batch_info['batch_id']):
        jf = id_to_file.get(result.custom_id, f"{result.custom_id}.json")
        if result.result.type == 'succeeded':
            raw = result.result.message.content[0].text
            raw = re.sub(r'^```json\s*', '', raw.strip())
            raw = re.sub(r'\s*```$', '', raw.strip())
            try:
                parsed = json.loads(raw)
                with open(os.path.join(FEATURES_DIR, jf), 'w') as f:
                    json.dump(parsed, f, indent=2)
                saved += 1
            except json.JSONDecodeError:
                with open(os.path.join(FEATURES_DIR, jf.replace('.json', '.raw.md')), 'w') as f:
                    f.write(raw)
                errors += 1
        else:
            errors += 1
        if (saved + errors) % 1000 == 0:
            print(f"  {saved + errors}...")

    print(f"\nDone: {saved} saved, {errors} errors")

    # Show samples
    for s in sorted([f for f in os.listdir(FEATURES_DIR) if f.endswith('.json')])[:3]:
        with open(os.path.join(FEATURES_DIR, s)) as f:
            print(f"\n--- {s} ---")
            print(json.dumps(json.load(f), indent=2))